In [3]:
# Run this first
# !pip install ipywidgets pandas

In [4]:
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, clear_output

INPUT_FILE = "candidates_list.csv"
df = pd.read_csv(INPUT_FILE)

# ── Build constituency list ───────────────────────────────────────────────
combos = (
    df[["EnglishDistrictName", "DistrictName", "ConstName"]]
    .drop_duplicates()
    .sort_values(["EnglishDistrictName", "ConstName"])
    .reset_index(drop=True)
)

# Count candidates per constituency
counts = df.groupby(["EnglishDistrictName", "ConstName"]).size().reset_index(name="count")
combos = combos.merge(counts, on=["EnglishDistrictName", "ConstName"])

# Build checkbox labels
options = [
    f"{row['EnglishDistrictName'].title()} — Constituency {row['ConstName']}  ({row['count']} candidates)"
    for _, row in combos.iterrows()
]

print(f"📋 {len(df)} candidates  |  {len(combos)} constituencies loaded from {INPUT_FILE}")

📋 492 candidates  |  19 constituencies loaded from candidates_list.csv


In [7]:
# ── Filter UI ─────────────────────────────────────────────────────────────

# District filter dropdown
districts = ["All districts"] + sorted(combos["EnglishDistrictName"].str.title().unique().tolist())
district_dd = widgets.Dropdown(
    options=districts,
    description="District:",
    style={"description_width": "80px"},
    layout=widgets.Layout(width="280px")
)

# Search box
search_box = widgets.Text(
    placeholder="Search constituency or district...",
    description="Search:",
    style={"description_width": "80px"},
    layout=widgets.Layout(width="320px")
)

# Select all / none buttons
btn_all  = widgets.Button(description="✅ Select All",  button_style="success", layout=widgets.Layout(width="130px"))
btn_none = widgets.Button(description="❌ Select None", button_style="danger",  layout=widgets.Layout(width="130px"))

# Checkboxes — one per constituency
checkboxes = [
    widgets.Checkbox(value=False, description=opt, indent=False,
                     layout=widgets.Layout(width="500px"))
    for opt in options
]
checkbox_box = widgets.VBox(checkboxes)

# Counter label
counter_label = widgets.HTML(value="<b>0</b> constituencies selected")

# Confirm button
btn_confirm = widgets.Button(
    description="🚀 Confirm Selection",
    button_style="primary",
    layout=widgets.Layout(width="200px")
)
confirm_out = widgets.Output()

# ── Logic ─────────────────────────────────────────────────────────────────
def update_counter(*_):
    n = sum(cb.value for cb in checkboxes)
    total_cands = sum(
        combos.iloc[i]["count"]
        for i, cb in enumerate(checkboxes) if cb.value
    )
    counter_label.value = f"<b>{n}</b> constituencies selected — <b>{total_cands}</b> candidates total"

def apply_filters(*_):
    search = search_box.value.strip().lower()
    district = district_dd.value
    for i, cb in enumerate(checkboxes):
        row = combos.iloc[i]
        dist_match = (district == "All districts") or (row["EnglishDistrictName"].title() == district)
        text_match = (search == "") or (search in options[i].lower())
        cb.layout.display = "" if (dist_match and text_match) else "none"

def select_all(_):
    for cb in checkboxes:
        if cb.layout.display != "none":
            cb.value = True

def select_none(_):
    for cb in checkboxes:
        cb.value = False

def confirm(_):
    with confirm_out:
        clear_output()
        selected_rows = [combos.iloc[i] for i, cb in enumerate(checkboxes) if cb.value]
        if not selected_rows:
            print("⚠️  No constituencies selected!")
            return

        global selected_df
        mask = df.apply(
            lambda r: any(
                r["EnglishDistrictName"] == sr["EnglishDistrictName"] and
                r["ConstName"] == sr["ConstName"]
                for sr in selected_rows
            ), axis=1
        )
        selected_df = df[mask].reset_index(drop=True)

        print(f"✅ Locked in! {len(selected_rows)} constituencies, {len(selected_df)} candidates.")
        print(f"   → Use `selected_df` in the next cell to run the scraper.\n")
        print("Selected:")
        for sr in selected_rows:
            cands = combos[(combos["EnglishDistrictName"] == sr["EnglishDistrictName"]) &
                           (combos["ConstName"] == sr["ConstName"])]["count"].values[0]
            print(f"   • {sr['EnglishDistrictName'].title()} — Constituency {sr['ConstName']}  ({cands} candidates)")

# Wire up events
for cb in checkboxes:
    cb.observe(update_counter, names="value")
district_dd.observe(apply_filters, names="value")
search_box.observe(apply_filters, names="value")
btn_all.on_click(select_all)
btn_none.on_click(select_none)
btn_confirm.on_click(confirm)

# ── Layout ────────────────────────────────────────────────────────────────
display(widgets.VBox([
    widgets.HTML("<h3>🗳️ Nepal Election — Constituency Picker</h3>"),
    widgets.HBox([district_dd, search_box]),
    widgets.HBox([btn_all, btn_none]),
    widgets.HTML("<hr style='margin:6px 0'>"),
    checkbox_box,
    widgets.HTML("<hr style='margin:6px 0'>"),
    counter_label,
    btn_confirm,
    confirm_out,
]))

In [6]:
# ── Preview selected candidates ───────────────────────────────────────────
# Run after confirming above

print(f"Total candidates to scrape: {len(selected_df)}")
selected_df.groupby(["EnglishDistrictName", "ConstName"]).size().reset_index(name="Candidates")

NameError: name 'selected_df' is not defined

In [ ]:
# ── Run scraper on selected_df ────────────────────────────────────────────
# selected_df is passed directly — no file re-read needed

# (paste your scraper functions here, or import from election_scraper.py)
# from election_scraper import search_candidate, candidate_path, already_scraped
#
# import pandas as pd
# from pathlib import Path
# from datetime import datetime
#
# OUTPUT_DIR = "data"
# done = 0
#
# for idx, row in selected_df.iterrows():
#     name_en = row["EnglishCandidateName"]
#     dist_en = row["EnglishDistrictName"]
#     const   = row["ConstName"]
#     out_path = candidate_path(dist_en, const, name_en)
#
#     if already_scraped(out_path):
#         print(f"⏭  Skip {name_en}")
#         continue
#
#     articles = search_candidate(row)
#     out_path.parent.mkdir(parents=True, exist_ok=True)
#     pd.DataFrame(articles).to_csv(out_path, index=False)
#     done += 1
#     print(f"✅ [{done}] {name_en} — {len(articles)} articles")

print("Uncomment the block above and import your scraper functions to run.")